# What does this Script do?

## Imports the Relevant data from the Partners in care Looker database, cleans and combines it, then updates a SQl table where its used to maintain a PowerBI Dashboard.  

import python packages

In [1]:
import pandas as pd
import os
import json
import datetime
from datetime import datetime, date, timezone
import numpy as np
import pytz


from io import StringIO
import looker_sdk

import custom packages

In [2]:
from utils.error_handling import Task, TaskManager

# general utilities for all payment predictions
from utils.email_utils import send_task_alert
from utils.ds_logger_threaded import get_mt_ds_logger
from ds_sql_connections import GsAutomationReliefAssistanceConnection

initialize task tracker

In [3]:
# initialization of the task manager with the overall name of the task this code is supposed to do.
task_mgr = TaskManager("partners in care clarity looker data ingestion")

initialize logger

In [4]:
task = Task(task_nm="Initialize log file", task_details="")
try:
    # creating any relevant dates
    now = datetime.now()
    now_str = now.strftime("%Y%m%d_%H%M%S")
    # log file info
    logging_dir = os.path.join(os.getcwd(), "logs")
    logging_filename = "partners_in_care_clarity_looker_data_ingestion_{}.log".format(
        now_str
    )
    logging_filename_with_path = os.path.join(logging_dir, logging_filename)

    if not os.path.exists(logging_dir):
        os.mkdir(logging_dir)

    logger = get_mt_ds_logger(
        logging_filename_with_path,
        "Partners_In_Care",
        "Data_ingestion",
        level="INFO",
        format_string="%(asctime)s %(levelname)s %(message)s",
    )
    # updating the task tracker
    task["Details"] = "Successfully created and began writing to log file"
    task["Success"] = True
    logger.info("Data ingestion for partners in care started")

except Exception as e:
    task["Details"] = str(e)
    print(e)

task_mgr.add_task(task)

initialize variables

In [5]:
task = Task(task_nm="Initialize starting global variables", task_details="")
try:
    task_mgr.check_errors()
    main_directory_path = str(os.getcwd())
    config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
    looker_config_path = main_directory_path + "\\utils\\looker.ini"

    # variables used in pulling records from looker
    model = "oahu_connection_model"
    view = "base"
    limit = 100000

    # creating the config object to pull the authentication credentials later in various calls.
    with open(config_path, "r") as file:
        config_object = json.load(file)

    sql_username = config_object["Josh"]["username"]
    sql_password = config_object["Josh"]["password"]

    today = date.today()
    today_str = today.strftime("%m_%d_%Y")
    reporting_period_today_str = today.strftime("%Y-%m-%d")

    now_utc = datetime.now(timezone.utc)
    hst = pytz.timezone("Pacific/Honolulu")
    now_hst = datetime.now(hst)
    now_utc_str = now_utc.strftime("%Y/%m/%d %H:%M:%S")
    now_str_formatted = now.strftime("%Y/%m/%d %H:%M:%S")
    now_hst_str = now_hst.strftime("%Y/%m/%d %H:%M:%S")

    # enrollment filter ranges need to fit looker filter conventions
    date_ranges = [
        "this quarter",
        "last quarter",
        "this year",
        "last year",
        "90 day ago for 90 day",
        "3 day ago for 1 day",
    ]
    labels = {
        "this quarter": "this quarter",
        "last quarter": "last quarter",
        "this year": "this year",
        "last year": "last year",
        "90 day ago for 90 day": "last 90 days",
        "3 day ago for 1 day": "3 days ago",
    }

    # updating the task tracker
    task["Details"] = "successfully initialize global variables"
    task["Success"] = True
    logger.info("Initialized starting global variables")
except Exception as e:
    task["Details"] = str(e)
    logger.error(f"Initializing global variables failed with the following error :{e}")
    print(e)

task_mgr.add_task(task)

Grab program level information 

In [6]:
task = Task(
    task_nm="Connect to database and ingest program level data for various time periods",
    task_details="",
)
try:
    task_mgr.check_errors()
    # initialize sdk
    sdk = looker_sdk.init40(looker_config_path)
    # Define fields
    program_fields = [
        "agencies.name",
        "agencies.victim_service_provider",
        "clients.count",
        "enrollments.average_duration",
        "enrollments.count",
        "primary_sites.coc",
        "programs.continuum_project",
        "programs.name",
        "programs.project_type_code",
        "programs.availability_start_date",
        "programs.availability_end_date",
        "programs.status",
        "programs.id",
        "enrollments.reporting_period_start_date",
        "enrollments.reporting_period_end_date",
        "programs.ref_target_b",
        "program_hmis_participation_statuses.hmis_participation_status",
        "program_inventory.total_bed_inventory",
        "enrollments.total_days_in_project_during_reporting_period",
    ]

    # Function to generate query body
    def generate_body(date_filter, model, view, program_fields, limit):
        return {
            "model": model,
            "view": view,
            "fields": program_fields,
            "filters": {
                "enrollments.date_filter": date_filter,
                "program_inventory.end_date": "null",
            },
            "limit": limit,
        }

    # Function to run the query
    def run_query(date_filter, sdk, model, view, program_fields, limit):
        body = generate_body(date_filter, model, view, program_fields, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results (add Reporting Period column)
    def process_result(result, date_filter):
        df = pd.read_csv(StringIO(result))
        df["Reporting Period"] = labels[date_filter]
        df.rename(
            columns={
                "Clients Count Client IDs": "Clients Enrolled During the Reporting Period",
                "Enrollments Count": "Total Enrollments During the Reporting Period",
            },
            inplace=True,
        )
        return df

    # Collect all DataFrames
    all_dataframes = []

    for date_filter in date_ranges:
        # Run query and process results for each date range
        result = run_query(date_filter, sdk, model, view, program_fields, limit)
        df = process_result(result, date_filter)
        all_dataframes.append(df)

    # Concatenate all DataFrames into one
    program_level_df = pd.concat(all_dataframes)

    # Now, program_level_df holds the combined data for all periods.

    # updating the task tracker
    task["Details"] = "Successfully grabbed program data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled program data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing program data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Pull bed inventory and calculate number of bed available during reporting period

In [7]:
task = Task(
    task_nm="Pull Bed inventory, calculate days in reporting period and get beds available during reporting period",
    task_details="",
)
try:
    task_mgr.check_errors()

    # Define fields
    bed_inventory_fields = [
        "programs.id",
        "enrollments.reporting_period_start_date",
        "enrollments.reporting_period_end_date",
        "program_inventory.start_date",
        "program_inventory.end_date",
        "program_inventory.total_bed_inventory",
    ]

    # Function to generate query body
    def generate_body(date_filter, model, view, bed_inventory_fields, limit):
        return {
            "model": model,
            "view": view,
            "fields": bed_inventory_fields,
            "filters": {
                "enrollments.date_filter": date_filter,
            },
            "limit": limit,
        }

    # Function to run the query
    def run_query(date_filter, sdk, model, view, bed_inventory_fields, limit):
        body = generate_body(date_filter, model, view, bed_inventory_fields, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results (add Reporting Period column)
    def process_result(result, date_filter):
        df = pd.read_csv(StringIO(result))

        df["Enrollments Reporting Period Start Date"] = pd.to_datetime(
            df["Enrollments Reporting Period Start Date"]
        ).copy()
        df["Enrollments Reporting Period End Date"] = pd.to_datetime(
            df["Enrollments Reporting Period End Date"]
        ).copy()
        df["Bed and Unit Inventory Start Date"] = pd.to_datetime(
            df["Bed and Unit Inventory Start Date"]
        ).copy()
        df["Bed and Unit Inventory End Date"] = pd.to_datetime(
            df["Bed and Unit Inventory End Date"]
        ).copy()
        today = pd.Timestamp(datetime.now().date())
        df["Bed and Unit Inventory End Date"] = (
            df["Bed and Unit Inventory End Date"].fillna(today).copy()
        )
        df = df[~df["Bed and Unit Inventory Start Date"].isna()].copy()
        df["overlap_days"] = (
            pd.to_datetime(
                df[
                    [
                        "Enrollments Reporting Period End Date",
                        "Bed and Unit Inventory End Date",
                    ]
                ].min(axis=1)
            )
            - pd.to_datetime(
                df[
                    [
                        "Enrollments Reporting Period Start Date",
                        "Bed and Unit Inventory Start Date",
                    ]
                ].max(axis=1)
            )
        ).dt.days + 1

        # Ensure non-negative overlaps (if no overlap, set to 0)
        df["overlap_days"] = df["overlap_days"].apply(lambda x: max(0, x))
        df["Bed and Unit Inventory Total Bed Inventory for Reporting Period"] = (
            df["Bed and Unit Inventory Total Bed Inventory"] * df["overlap_days"]
        )
        df = df.pivot_table(
            index="Programs Program ID",
            values="Bed and Unit Inventory Total Bed Inventory for Reporting Period",
            aggfunc="sum",
        ).reset_index(drop=False)
        df["Reporting Period"] = labels[date_filter]

        return df

    # Collect all DataFrames
    all_dataframes = []

    for date_filter in date_ranges:
        # Run query and process results for each date range
        result = run_query(date_filter, sdk, model, view, bed_inventory_fields, limit)
        df = process_result(result, date_filter)
        all_dataframes.append(df)

    # Concatenate all DataFrames into one
    total_inventory_df = pd.concat(all_dataframes)

    # updating the task tracker
    task["Details"] = (
        "Successfully pulled Bed inventory, calculated days in reporting period and got beds available during reporting period"
    )
    task["Success"] = True
    logger.info(
        "Pulled Bed inventory, calculated days in reporting period and got beds available during reporting period"
    )

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"Pulling Bed inventory, calculating days in reporting period and getting beds available during reporting period failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

pull program level counts with and without move in date

In [8]:
task = Task(
    task_nm="Connect to database and ingest count program level data for various time periods",
    task_details="",
)
try:
    task_mgr.check_errors()
    # Define fields and limit
    move_in_counts_fields = [
        "programs.id",
        "enrollments.average_duration",
        "clients.count",
    ]

    # Function to generate query body
    def generate_body(date_filter, move_in_status, model, view, limit):
        move_in_filter = "not null" if move_in_status == "with" else "null"
        return {
            "model": model,
            "view": view,
            "fields": [move_in_counts_fields],
            "filters": {
                "enrollments.date_filter": date_filter,
                "program_inventory.end_date": "null",
                "household_move_in_date.move_in_date": move_in_filter,
            },
            "limit": limit,
        }

    # Function to run the query
    def run_query(date_filter, move_in_status, sdk, model, view, limit):
        body = generate_body(date_filter, move_in_status, model, view, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process results (rename columns)
    def process_result(result, move_in_status):
        df = pd.read_csv(StringIO(result))
        suffix = "With Move In" if move_in_status == "with" else "Without Move In"
        df.rename(
            columns={
                "Enrollments Average Days in Project": f"Enrollments Average Days in Project {suffix}",
                "Clients Count Client IDs": f"Clients Enrolled During the Reporting Period {suffix}",
            },
            inplace=True,
        )
        return df

    move_in_statuses = ["with", "without"]
    reporting_period_labels = {
        "this quarter": "This Quarter",
        "last quarter": "Last Quarter",
        "this year": "This Year",
        "last year": "Last Year",
    }

    # Collect all DataFrames
    all_dataframes = []

    for date_filter in date_ranges:
        # Run queries for both 'with' and 'without' move-ins
        for move_in_status in move_in_statuses:
            result = run_query(date_filter, move_in_status, sdk, model, view, limit)
            df = process_result(result, move_in_status)
            if move_in_status == "with":
                with_move_in_df = df
            else:
                without_move_in_df = df

        # Merge the 'with' and 'without' DataFrames
        merged_df = pd.merge(
            with_move_in_df, without_move_in_df, how="outer", on="Programs Program ID"
        )
        merged_df["Reporting Period"] = labels[date_filter]
        all_dataframes.append(merged_df)

    # Concatenate all dataframes into one
    final_counts_df = pd.concat(all_dataframes)
    # updating the task tracker
    task["Details"] = "Successfully grabbed count program data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled count program data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing count program data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

pull enrollments

In [9]:
task = Task(
    task_nm="Connect to database and ingest enrollment level data for various time periods",
    task_details="",
)
try:
    task_mgr.check_errors()

    # Define fields
    enrollment_fields = [
        "programs.name",
        "programs.id",
        "enrollments.id",
        "enrollments.start_date",
        "household_move_in_date.move_in_date",
    ]

    # Function to generate query body
    def generate_body(date_filter, model, view, enrollment_fields, limit):
        return {
            "model": model,
            "view": view,
            "fields": enrollment_fields,
            "filters": {
                "enrollments.date_filter": date_filter,
                "enrollments.still_in_program": "Yes",
            },
            "limit": limit,
        }

    # Function to run the query
    def run_query(date_filter, sdk, model, view, enrollment_fields, limit):
        body = generate_body(date_filter, model, view, enrollment_fields, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    def generate_condition(df, day_threshold, move_in_status):
        # Define the category based on the threshold
        day_category = f"day_category_{day_threshold}"

        # Create a condition for days (e.g., "over 90 days" or "less than 90 days")
        if move_in_status == "with":
            move_in_condition = df["Enrollments Household Move-In Date"].notna()
        else:
            move_in_condition = df["Enrollments Household Move-In Date"].isna()

        # Define if it's "over" or "less than" based on the threshold
        return (
            (df[day_category] == f"less than {day_threshold} days") & move_in_condition,
            (df[day_category] == f"over {day_threshold} days") & move_in_condition,
        )

    # Function to process the results (add Reporting Period column)
    def process_result(result, date_filter):
        df = pd.read_csv(StringIO(result))
        # Convert 'Enrollments Project Start Date' to datetime
        df["Enrollments Project Start Date"] = pd.to_datetime(
            df["Enrollments Project Start Date"]
        ).copy()

        # Calculate 'Days Since Start'
        df["Days Since Start"] = (
            datetime.now() - df["Enrollments Project Start Date"]
        ).dt.days.astype(int)

        # Create day categories based on 'Days Since Start'
        df["day_category_90"] = df["Days Since Start"].apply(
            lambda x: "over 90 days" if x > 90 else "less than 90 days"
        )
        df["day_category_180"] = df["Days Since Start"].apply(
            lambda x: "over 180 days" if x > 180 else "less than 180 days"
        )

        # Initialize a dictionary to store results
        results = {}

        # Define thresholds and move-in categories
        thresholds = [90, 180]
        move_in_categories = ["with", "without"]

        # Iterate over day thresholds and move-in statuses to generate conditions
        for threshold in thresholds:
            for move_in_status in move_in_categories:
                # Generate the two conditions (less than and over)
                less_than_condition, over_condition = generate_condition(
                    df, threshold, move_in_status
                )

                # Create keys for each condition to store results
                less_than_key = f"Active Enrollments Open For Less Than {threshold} Days {move_in_status.capitalize()} Move In Date"
                over_key = f"Active Enrollments Open For More Than {threshold} Days {move_in_status.capitalize()} Move In Date"

                # Apply conditions and count occurrences by 'Programs Program ID'
                results[less_than_key] = (
                    df[less_than_condition].groupby("Programs Program ID").size()
                )
                results[over_key] = (
                    df[over_condition].groupby("Programs Program ID").size()
                )

        # Convert the results into a DataFrame and handle missing values
        result_df = pd.DataFrame(results).fillna(0).astype(int).reset_index()

        result_df["Reporting Period"] = labels[date_filter]

        return result_df

    # Collect all DataFrames
    all_dataframes = []

    for date_filter in date_ranges:
        # Run query and process results for each date range
        result = run_query(date_filter, sdk, model, view, enrollment_fields, limit)
        df = process_result(result, date_filter)
        all_dataframes.append(df)

    # Concatenate all DataFrames into one
    enrollment_level_df = pd.concat(all_dataframes)

    # updating the task tracker
    task["Details"] = "Successfully grabbed enrollment data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled enrollment data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing enrollment data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

merging all pulled data and calculated any extra fields

In [10]:
task = Task(
    task_nm="merging data and creating extra fields",
    task_details="",
)
try:
    task_mgr.check_errors()

    program_level_with_inventory_totals_df = program_level_df.merge(
        total_inventory_df, how="left", on=["Programs Program ID", "Reporting Period"]
    )

    program_level_with_counts_df = program_level_with_inventory_totals_df.merge(
        final_counts_df, how="left", on=["Programs Program ID", "Reporting Period"]
    )
    final_results_df = pd.merge(
        program_level_with_counts_df,
        enrollment_level_df,
        how="left",
        on=["Programs Program ID", "Reporting Period"],
    )

    final_results_df["Program Utilization During The Reporting Period"] = np.where(
        final_results_df[
            "Bed and Unit Inventory Total Bed Inventory for Reporting Period"
        ],
        final_results_df[
            "Enrollments Total Days in Project During the Reporting Period"
        ]
        / final_results_df[
            "Bed and Unit Inventory Total Bed Inventory for Reporting Period"
        ],
        np.nan,
    ).round(4)

    final_results_df[
        [
            "Active Enrollments Open For Less Than 90 Days With Move In Date",
            "Active Enrollments Open For Less Than 90 Days Without Move In Date",
            "Active Enrollments Open For More Than 90 Days With Move In Date",
            "Active Enrollments Open For More Than 90 Days Without Move In Date",
            "Active Enrollments Open For Less Than 180 Days With Move In Date",
            "Active Enrollments Open For Less Than 180 Days Without Move In Date",
            "Active Enrollments Open For More Than 180 Days With Move In Date",
            "Active Enrollments Open For More Than 180 Days Without Move In Date",
            "Enrollments Average Days in Project With Move In",
            "Clients Enrolled During the Reporting Period With Move In",
            "Enrollments Average Days in Project Without Move In",
            "Clients Enrolled During the Reporting Period Without Move In",
        ]
    ] = (
        final_results_df[
            [
                "Active Enrollments Open For Less Than 90 Days With Move In Date",
                "Active Enrollments Open For Less Than 90 Days Without Move In Date",
                "Active Enrollments Open For More Than 90 Days With Move In Date",
                "Active Enrollments Open For More Than 90 Days Without Move In Date",
                "Active Enrollments Open For Less Than 180 Days With Move In Date",
                "Active Enrollments Open For Less Than 180 Days Without Move In Date",
                "Active Enrollments Open For More Than 180 Days With Move In Date",
                "Active Enrollments Open For More Than 180 Days Without Move In Date",
                "Enrollments Average Days in Project With Move In",
                "Clients Enrolled During the Reporting Period With Move In",
                "Enrollments Average Days in Project Without Move In",
                "Clients Enrolled During the Reporting Period Without Move In",
            ]
        ]
        .fillna(0)
        .copy()
    )
    final_results_df[
        [
            "Active Enrollments Open For Less Than 90 Days With Move In Date",
            "Active Enrollments Open For Less Than 90 Days Without Move In Date",
            "Active Enrollments Open For More Than 90 Days With Move In Date",
            "Active Enrollments Open For More Than 90 Days Without Move In Date",
            "Active Enrollments Open For Less Than 180 Days With Move In Date",
            "Active Enrollments Open For Less Than 180 Days Without Move In Date",
            "Active Enrollments Open For More Than 180 Days With Move In Date",
            "Active Enrollments Open For More Than 180 Days Without Move In Date",
            "Enrollments Average Days in Project With Move In",
            "Clients Enrolled During the Reporting Period With Move In",
            "Enrollments Average Days in Project Without Move In",
            "Clients Enrolled During the Reporting Period Without Move In",
        ]
    ] = (
        final_results_df[
            [
                "Active Enrollments Open For Less Than 90 Days With Move In Date",
                "Active Enrollments Open For Less Than 90 Days Without Move In Date",
                "Active Enrollments Open For More Than 90 Days With Move In Date",
                "Active Enrollments Open For More Than 90 Days Without Move In Date",
                "Active Enrollments Open For Less Than 180 Days With Move In Date",
                "Active Enrollments Open For Less Than 180 Days Without Move In Date",
                "Active Enrollments Open For More Than 180 Days With Move In Date",
                "Active Enrollments Open For More Than 180 Days Without Move In Date",
                "Enrollments Average Days in Project With Move In",
                "Clients Enrolled During the Reporting Period With Move In",
                "Enrollments Average Days in Project Without Move In",
                "Clients Enrolled During the Reporting Period Without Move In",
            ]
        ]
        .astype(int)
        .copy()
    )

    final_results_df["Enrollments Reporting Period End Date"] = pd.to_datetime(
        final_results_df["Enrollments Reporting Period End Date"], format="%Y-%m-%d"
    )

    # Replace any dates after today with today's date
    final_results_df["Enrollments Reporting Period End Date"] = final_results_df[
        "Enrollments Reporting Period End Date"
    ].apply(
        lambda x: (
            reporting_period_today_str
            if x > pd.Timestamp(reporting_period_today_str)
            else x
        )
    )

    # Convert the column back to string
    final_results_df["Enrollments Reporting Period End Date"] = final_results_df[
        "Enrollments Reporting Period End Date"
    ].astype(str)

    # updating the task tracker
    task["Details"] = "Successfully merged data and generated extra fields"
    task["Success"] = True
    logger.info("Merged data and generated extra fields")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"Merging data and generating extra fields failed with teh following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

upload to database

In [ ]:
task = Task(
    task_nm="uploading new table to database",
    task_details="",
)
try:
    task_mgr.check_errors()

    final_results_df["Update Time HST"] = now_hst_str
    final_results_df["Update Time CST"] = now_str_formatted
    final_results_df["Update Time UTC"] = now_utc_str

    conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)

    conn.insert_with_progress(
        df=final_results_df,
        schema="dbo",
        tbl_nm="partners_in_care_length_of_stay_and_utilization",
        dtypes={},
    )

    # updating the task tracker
    task["Details"] = "Successfully updated data in database"
    task["Success"] = True
    logger.info("Updated data in Database")

except Exception as e:
    task["Details"] = str(e)
    logger.error(f"uploading new data to database failed with teh following error:{e}")
    print(e)

task_mgr.add_task(task)

In [25]:
logger.cleanup()

In [26]:
if task_mgr._process_success != "Success":
    subject, body = task_mgr.generate_task_email(today_str)

    send_task_alert(subject, body)